In [2]:

!pip install -q -U google-genai

import sqlite3
from google import genai
from google.genai import types



In [3]:

from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()

api_key = user_secrets.get_secret("GEMINI_API_KEY")

client = genai.Client(api_key=api_key)



In [4]:

response = client.models.generate_content(
    model="gemini-3.1-flash-lite",
    contents="Say hello and explain in one sentence what you are."
)

print(response.text)



Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Hello! I am a large language model, trained by Google to assist you with a wide range of tasks by processing and generating information.


In [5]:
import sqlite3

connection = sqlite3.connect(":memory:")
cursor = connection.cursor()

cursor.execute("""
CREATE TABLE customers (
    id INTEGER PRIMARY KEY,
    name TEXT,
    country TEXT
)
""")

customers = [
    (1, "Ali", "Algeria"),
    (2, "Sarah", "France"),
    (3, "John", "USA"),
    (4, "Emma", "Germany")
]

cursor.executemany(
    "INSERT INTO customers (id, name, country) VALUES (?, ?, ?)",
    customers
)

connection.commit()

cursor.execute("SELECT * FROM customers")

rows = cursor.fetchall()

print(rows)


[(1, 'Ali', 'Algeria'), (2, 'Sarah', 'France'), (3, 'John', 'USA'), (4, 'Emma', 'Germany')]


In [8]:

def get_customers():
    cursor.execute("SELECT * FROM customers")
    return cursor.fetchall()

print(get_customers())



[(1, 'Ali', 'Algeria'), (2, 'Sarah', 'France'), (3, 'John', 'USA'), (4, 'Emma', 'Germany')]


In [9]:
tool = types.Tool(
    function_declarations=[
        types.FunctionDeclaration(
            name="get_customers",
            description="Get all customers from the customers database.",
            parameters={
                "type": "object",
                "properties": {}
            }
        )
    ]
)



In [10]:
response = client.models.generate_content(
    model="gemini-3.1-flash-lite",
    contents="Show me all the customers.",
    config=types.GenerateContentConfig(
        tools=[tool]
    )
)

print(response)



sdk_http_response=HttpResponse(
  headers=<dict len=11>
) candidates=[Candidate(
  content=Content(
    parts=[
      Part(
        function_call=FunctionCall(
          args={},
          id='call_3100753',
          name='get_customers'
        ),
        thought_signature=b'\x12q\no\x01\x11M2\x0f\xde7\x1d*\x8d=<\xba\x86NJ\xa7A\x16\xd8t\xae\xd0\xbb|\xfb\xd3\xd6_\x92\xbb\xe6\xce\x8f:\xc4bJ\x97\xa8\xdf2\xfc\xcb\x1e7\x1e$\x072\xbdM\x83\xaeC\xa4\x93\xbb\x95\x83&\x85:\x98lwv\x9d\x1d(\xc9\x9a\xf9@\xa1\xc3\xf6\x9cjh\x01\xcf\x85\xd8\x1e?\x80@Fd\x08\x86]...'
      ),
    ],
    role='model'
  ),
  finish_reason=<FinishReason.STOP: 'STOP'>,
  index=0
)] create_time=None model_version='gemini-3.1-flash-lite' prompt_feedback=None response_id='zyybaoCcHpWFz7IPvu2JqAo' usage_metadata=GenerateContentResponseUsageMetadata(
  candidates_token_count=10,
  prompt_token_count=36,
  prompt_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=36
    ),
  

In [11]:
part = response.candidates[0].content.parts[0]

function_call = part.function_call

print("Function:", function_call.name)
print("Arguments:", function_call.args)
print("Call ID:", function_call.id)



Function: get_customers
Arguments: {}
Call ID: call_3100753


In [12]:
if function_call.name == "get_customers":
    result = get_customers()

print(result)



[(1, 'Ali', 'Algeria'), (2, 'Sarah', 'France'), (3, 'John', 'USA'), (4, 'Emma', 'Germany')]


In [15]:
tool_response = types.Part.from_function_response(
    name=function_call.name,
    response={
        "customers": result
    }
)

final_response = client.models.generate_content(
    model="gemini-3.1-flash-lite",
    contents=[
        types.Content(
            role="user",
            parts=[
                types.Part(
                    text="Which customers are from Algeria?"
                )
            ]
        ),

        # Gemini's original function call
        response.candidates[0].content,

        # The result produced by our Python tool
        types.Content(
            role="user",
            parts=[tool_response]
        )
    ]
)

print(final_response.text)



The customer from Algeria is:

*   **Ali**


In [16]:
def execute_sql(query):
    cursor.execute(query)
    return cursor.fetchall()



In [17]:
result = execute_sql(
    "SELECT * FROM customers WHERE country = 'Algeria'"
)

print(result)



[(1, 'Ali', 'Algeria')]


In [18]:
sql_tool = types.Tool(
    function_declarations=[
        types.FunctionDeclaration(
            name="execute_sql",
            description=(
                "Execute a SQL query against the customers database "
                "and return the query results."
            ),
            parameters={
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "The SQL query to execute."
                    }
                },
                "required": ["query"]
            }
        )
    ]
)



In [19]:
question = "Which customers are from Algeria?"

response = client.models.generate_content(
    model="gemini-3.1-flash-lite",
    contents=question,
    config=types.GenerateContentConfig(
        tools=[sql_tool]
    )
)

print(response)



sdk_http_response=HttpResponse(
  headers=<dict len=11>
) candidates=[Candidate(
  content=Content(
    parts=[
      Part(
        function_call=FunctionCall(
          args={
            'query': "SELECT * FROM customers WHERE country = 'Algeria'"
          },
          id='call_3304443',
          name='execute_sql'
        ),
        thought_signature=b'\x12q\no\x01\x11M2\x0f\x9d\x14\xa20Z\xa1\x98\x1f+\x18\xe4\x03W\xd5g\n\xf4|\xd5\xf9Pz\xbaN\xc19<\x00\xc5\xf5\xb9`\\0S|\xda=\x91\xa8\x02\x8e\x80*\x87\xbb@\xf5u\xb7\x8d=fz`\xca\xda\xf2]\x113\\\xa7\xbd\x8c\xd0Z6b\x8c\x15\xc9\xbb\x9e\x86\x80V\xec\x04Is\xb0\xac;\xa9\x8e+\xef...'
      ),
    ],
    role='model'
  ),
  finish_reason=<FinishReason.STOP: 'STOP'>,
  index=0
)] create_time=None model_version='gemini-3.1-flash-lite' prompt_feedback=None response_id='aC6baq6_BP-Fz7IPnqzw8Ao' usage_metadata=GenerateContentResponseUsageMetadata(
  candidates_token_count=26,
  prompt_token_count=69,
  prompt_tokens_details=[
    ModalityTokenCount(

In [21]:
part = response.candidates[0].content.parts[0]

function_call = part.function_call

print("Function:", function_call.name)
print("Arguments:", function_call.args)



Function: execute_sql
Arguments: {'query': "SELECT * FROM customers WHERE country = 'Algeria'"}


In [23]:
Function: execute_sql

Arguments: {
    'query': "SELECT * FROM customers WHERE country = 'Algeria'"
}

In [24]:
query = function_call.args["query"]

print("SQL generated by Gemini:")
print(query)

result = execute_sql(query)

print("\nDatabase result:")
print(result)



SQL generated by Gemini:
SELECT * FROM customers WHERE country = 'Algeria'

Database result:
[(1, 'Ali', 'Algeria')]


In [26]:
tool_response = types.Part.from_function_response(
    name=function_call.name,
    response={
        "result": result
    }
)

final_response = client.models.generate_content(
    model="gemini-3.1-flash-lite",
    contents=[
        types.Content(
            role="user",
            parts=[
                types.Part(text=question)
            ]
        ),

        # Gemini's original function call
        response.candidates[0].content,

        # Result returned by our Python tool
        types.Content(
            role="user",
            parts=[tool_response]
        )
    ]
)

print(final_response.text)



The customer from Algeria is:

*   **Ali** (Customer ID: 1)


In [27]:
def run_agent(question):

    # 1. Start the conversation with the user's question
    contents = [
        types.Content(
            role="user",
            parts=[
                types.Part(text=question)
            ]
        )
    ]

    # 2. Keep running until Gemini gives us a final answer
    while True:

        # 3. Ask Gemini what should happen next
        response = client.models.generate_content(
            model="gemini-3.1-flash-lite",
            contents=contents,
            config=types.GenerateContentConfig(
                tools=[sql_tool]
            )
        )

        # 4. Save Gemini's response in the conversation
        contents.append(
            response.candidates[0].content
        )

        # 5. Look for a function call
        function_call = None

        for part in response.candidates[0].content.parts:

            if part.function_call:
                function_call = part.function_call
                break

        # 6. No function call = Gemini has finished
        if function_call is None:
            return response.text

        # 7. Check which tool Gemini requested
        if function_call.name == "execute_sql":

            # 8. Get the SQL generated by Gemini
            query = function_call.args["query"]

            print("Gemini generated:")
            print(query)

            # 9. Execute the SQL
            try:
                result = execute_sql(query)

                print("Database result:")
                print(result)

            except Exception as error:

                # 10. If SQL fails, capture the error
                result = {
                    "error": str(error)
                }

                print("SQL ERROR:")
                print(error)

            # 11. Convert our result into a Gemini function response
            tool_response = types.Part.from_function_response(
                name=function_call.name,
                response={
                    "result": result
                }
            )

            # 12. Send the result/error back to Gemini
            contents.append(
                types.Content(
                    role="user",
                    parts=[
                        tool_response
                    ]
                )
            )







In [28]:
answer = run_agent(
    "Which customers are from Algeria?"
)

print("\nFinal answer:")
print(answer)

Gemini generated:
SELECT * FROM customers WHERE country = 'Algeria';
Database result:
[(1, 'Ali', 'Algeria')]

Final answer:
The customer from Algeria is:

*   **ID:** 1
*   **Name:** Ali


In [31]:
## 🧪 Now test something harder

answer = run_agent(
    "How many customers are in the database?"
)

print("\nFinal answer:")
print(answer)






Gemini generated:
SELECT COUNT(*) FROM customers;
Database result:
[(4,)]

Final answer:
There are 4 customers in the database.


In [32]:
answer = run_agent(
    "What are the names of customers who are not from Algeria?"
)

print("\nFinal answer:")
print(answer)

Gemini generated:
SELECT name FROM customers WHERE country != 'Algeria'
Database result:
[('Sarah',), ('John',), ('Emma',)]

Final answer:
The names of the customers who are not from Algeria are Sarah, John, and Emma.
